In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.model_selection import train_test_split

import joblib
import os
import scipy.sparse

%matplotlib inline
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

In [26]:
df = pd.read_csv('../data/fraudTrain.csv') 
test_df = pd.read_csv('../data/fraudTest.csv')
print("Shape:", df.shape)
df.head()

Shape: (1296675, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [27]:

DROP_COLS = ['Unnamed: 0', 'cc_num', 'first', 'last', 'street', 'trans_num']
DATETIME_COLS = ['trans_date_trans_time', 'dob']
NUMERIC_COLS = ['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long']
CATEGORICAL_COLS = ['merchant', 'category', 'gender', 'city', 'state', 'job']

TARGET = 'is_fraud'

In [28]:
def engineer_features(df):
    """Extract hour, day-of-week, month, and age from datetime columns."""
    df = df.copy()
    
    # Transaction time features
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['trans_hour'] = df['trans_date_trans_time'].dt.hour
    df['trans_day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['trans_month'] = df['trans_date_trans_time'].dt.month
    df['trans_is_weekend'] = df['trans_day_of_week'].isin([5, 6]).astype(int)
    
    # Age from date of birth
    df['dob'] = pd.to_datetime(df['dob'])
    df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
    
    # Drop original datetime columns
    df = df.drop(columns=['trans_date_trans_time', 'dob'])
    
    return df

In [29]:
df = engineer_features(df)

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

print("After engineering:", df.shape)
print("New columns:", df.columns.tolist())

After engineering: (1296675, 20)
New columns: ['merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'trans_hour', 'trans_day_of_week', 'trans_month', 'trans_is_weekend', 'age']


In [30]:
NUMERIC_COLS = ['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long',
                'trans_hour', 'trans_day_of_week', 'trans_month', 
                'trans_is_weekend', 'age']

CATEGORICAL_COLS = ['category', 'gender', 'state']

In [31]:
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df.columns]
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in df.columns]

print("Numeric:", NUMERIC_COLS)
print("\nCategorical:", CATEGORICAL_COLS)

Numeric: ['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long', 'trans_hour', 'trans_day_of_week', 'trans_month', 'trans_is_weekend', 'age']

Categorical: ['category', 'gender', 'state']


In [32]:
numerical_pipeline = Pipeline([('scalar',RobustScaler())])

In [33]:
categorical_pipeline = Pipeline([('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=True,      # KEY: keeps memory low
        min_frequency=100,       
        max_categories=50 ))])

In [34]:
preprocessor = ColumnTransformer([
    ('num',numerical_pipeline,NUMERIC_COLS),
    ('cat',categorical_pipeline,CATEGORICAL_COLS)
],remainder='drop',sparse_threshold=0.3 )

In [35]:
X = df.drop(TARGET,axis = 1)
y = df[TARGET]

In [36]:
X_train, X_test ,y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [37]:
print(X_train.shape,X_test.shape)

(1037340, 19) (259335, 19)


In [38]:
print(f"Train frauds: {y_train.sum()} ({y_train.mean()*100:.4f}%)")
print(f"Test frauds:  {y_test.sum()} ({y_test.mean()*100:.4f}%)")

Train frauds: 6005 (0.5789%)
Test frauds:  1501 (0.5788%)


In [39]:
# Fit on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print("Train shape:", X_train_processed.shape)
print("Test shape: ", X_test_processed.shape)
print("Data type:  ", type(X_train_processed))
print("Sparse?     ", hasattr(X_train_processed, 'toarray'))

Train shape: (1037340, 77)
Test shape:  (259335, 77)
Data type:   <class 'scipy.sparse._csr.csr_matrix'>
Sparse?      True


In [ ]:
scipy.sparse.save_npz('../data/X_train_v2.npz', X_train_processed)
scipy.sparse.save_npz('../data/X_test_v2.npz',  X_test_processed)
np.save('../data/y_train_v2.npy', y_train.values)
np.save('../data/y_test_v2.npy',  y_test.values)

joblib.dump(preprocessor, '../models/preprocessor_v2.pkl')

print("Saved sparse .npz files.")
print("X_train:", X_train_processed.shape)
print("X_test: ", X_test_processed.shape)

Saved sparse .npz files.
X_train: (1037340, 77)
X_test:  (259335, 77)


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from numeric
numeric_cols = [c for c in numeric_cols if c != 'is_fraud']

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"\nNumeric: {numeric_cols}")
print(f"\nCategorical: {categorical_cols}")

In [ ]:
def numeric_summary(df, cols):
    rows = []
    for col in cols:
        s = df[col].dropna()
        rows.append({
            'column': col,
            'count': len(s),
            'missing': df[col].isnull().sum(),
            'mean': s.mean(),
            'median': s.median(),
            'std': s.std(),
            'min': s.min(),
            'max': s.max(),
            'skewness': s.skew(),
            'kurtosis': s.kurtosis(),
            'unique': s.nunique()
        })
    return pd.DataFrame(rows).round(3)

num_summary = numeric_summary(df, numeric_cols)
num_summary = num_summary.sort_values('skewness', key=abs, ascending=False)
print(num_summary.to_string(index=False))

In [ ]:
n_cols = len(numeric_cols)
n_rows = (n_cols + 3) // 4   # 4 plots per row

fig, axes = plt.subplots(n_rows, 4, figsize=(18, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], bins=50, ax=axes[i], kde=True, color='steelblue')
    skew_val = df[col].skew()
    axes[i].set_title(f"{col}\nskew={skew_val:.2f}", fontsize=10)
    axes[i].set_xlabel("")

# Hide empty subplots
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()